# Proyecto Final — Derivados · Apple (AAPL): Valuación de un Bono Convertible Sintético
**Base metodológica:** caso *Molson Coors Brewing Co.: Valuing a Convertible Bond* (NE90-06-107)
**Formato de entrega:** dashboard tipo FINA 405 (tarjetas KPI + secciones), generado como HTML autocontenido
**Datos:** Refinitiv / LSEG (sesión ya abierta con tu App Key)

**Lógica del proyecto.** Apple no tiene convertibles en circulación; solo deuda *straight* senior unsecured.
Por eso replicamos el caso así: tomamos un bono real de Apple en Refinitiv como **bono puro** (su YTM es la tasa de
descuento, el equivalente al 5.75% de Molson) y preguntamos: *¿qué cupón tendría que pagar Apple si hoy emitiera ese
mismo bono como convertible a 6 años?*

| Paso | Caso Molson | Proyecto AAPL |
|---|---|---|
| 1 | Opción de conversión con Black-Scholes (Exhibit 2) | BS por acción y por bono, con griegas, con/sin dividendo |
| 2 | Cupón que deja el convertible a la par (Exhibit 3) | Cupón a la par usando YTM real del bono AAPL |
| 3 | Sensibilidad a la prima (15/25/40%) | Tabla y heatmap prima × volatilidad |
| 4 | Call spread: note hedge 25% + warrant 60% (Exhibit 4 / 1) | Mismo overlay, costo neto y "no free lunch" |
| 5 | Visión del CEO | Conclusiones automáticas con los números |
| + | — | Dashboard: curva de bonos AAPL, KPIs de empresa, contexto China |

> Ejecuta las celdas en orden. Si algún campo de Refinitiv no responde, el código lo marca como **⚠ respaldo** en el dashboard.

In [ ]:
# Si te falta alguna librería, descomenta:
# %pip install -q plotly scipy openpyxl
import numpy as np, pandas as pd, datetime as dt, warnings, os, json
from scipy.stats import norm
from scipy.optimize import brentq
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1 · Parámetros del proyecto

In [ ]:
AUTORES   = "Equipo 3"                       # edita
CURSO     = "Productos Derivados y Administración de Riesgos Corporativos · EGADE"
HOY       = dt.date.today()

TICKER_RIC = "AAPL.O"

# Bono recomendado (verificado contra el term sheet de Apple, SEC FWP may-2025)
BONO = dict(nombre="Apple Inc 4.500% Notes due 12-May-2032", cusip="037833FA3", isin="US037833FA32",
            cupon=0.045, vencimiento="2032-05-12", emision="2025-05-12", monto_usd=1_000_000_000,
            frecuencia=2, rango="Senior Unsecured", call="Make-whole; par call desde 12-Mar-2032")

# Universo de respaldo (solo se usa si la búsqueda en Refinitiv falla)
UNIVERSO_RESPALDO = [
    dict(nombre="AAPL 4.000% 10-May-2028", cusip="037833ET3", isin="US037833ET32", cupon=0.0400, vencimiento="2028-05-10"),
    dict(nombre="AAPL 4.150% 10-May-2030", cusip="037833EU0", isin="US037833EU05", cupon=0.0415, vencimiento="2030-05-10"),
    dict(nombre="AAPL 4.500% 12-May-2032", cusip="037833FA3", isin="US037833FA32", cupon=0.0450, vencimiento="2032-05-12"),
    dict(nombre="AAPL 3.350% 08-Aug-2032", cusip="037833EP1", isin="US037833EP10", cupon=0.0335, vencimiento="2032-08-08"),
]

# Diseño del convertible sintético (igual que el caso)
PLAZO_CONV      = 6.0     # años (caso: 6)
PRIMA_CONV      = 0.25    # caso: 25%
PRIMA_WARRANT   = 0.60    # caso: 60% (call spread)
AJUSTE_VOL_WARR = -0.02   # caso: warrant a 21% vs 23% (skew)
PRIMAS_SENS     = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]
VENTANA_VOL     = "1Y"    # "1Y", "3Y" o "5Y" (histórica)
VOL_MANUAL      = None    # p.ej. 0.26 si quieres usar vol implícita de Refinitiv (pantalla AAPL.O / opciones LEAPS)
USAR_DIVIDENDO  = True    # el caso usa BS sin dividendo; AAPL sí paga → Merton con q
MONTO_EMISION   = None    # None = mismo monto del bono elegido (USD 1,000M)

# Valores de respaldo — SOLO si Refinitiv no devuelve el dato (se marcan ⚠ en el dashboard)
RESPALDO = dict(S=250.0, precio_bono=100.0, ytm=0.045, div_yield=0.004, vol=0.27,
                ust={2:0.036, 3:0.037, 5:0.039, 7:0.041, 10:0.043, 30:0.047})
CARPETA = "."

## 2 · Conexión (reutiliza tu sesión abierta)

In [ ]:
API = None
g = globals()
if "rd" in g:            API = "rd"
elif "ld" in g:          rd = ld; API = "rd"          # lseg.data con alias ld
elif "ek" in g:          API = "ek"                    # eikon con ek.set_app_key(...) ya hecho
else:
    try:
        import refinitiv.data as rd
        try: rd.open_session()
        except Exception as e: print("open_session():", e)
        API = "rd"
    except ImportError:
        try:
            import eikon as ek; API = "ek"
        except ImportError:
            pass
print("API detectada:", API or "NINGUNA → modo respaldo")

FUENTES = {}   # registro de origen de cada dato (para el dashboard)

def get_data(instr, fields):
    if API is None: return pd.DataFrame()
    instr = [instr] if isinstance(instr, str) else list(instr)
    fields = [fields] if isinstance(fields, str) else list(fields)
    try:
        if API == "rd": df = rd.get_data(instr, fields)
        else:           df, _ = ek.get_data(instr, fields)
        return df if df is not None else pd.DataFrame()
    except Exception as e:
        print(f"  · sin dato {instr[:2]} {fields[:2]} → {str(e)[:90]}")
        return pd.DataFrame()

def first_value(instr, fields):
    """Prueba varios campos y devuelve el primer valor no vacío: (valor, campo)."""
    for f in fields:
        df = get_data(instr, [f])
        if not df.empty and df.shape[1] >= 2:
            v = df.iloc[0, -1]
            if v is not None and not (isinstance(v, float) and np.isnan(v)) and str(v).strip() not in ("", "<NA>", "nan"):
                return v, f
    return None, None

def num(v):
    try: return float(v)
    except Exception: return np.nan

def pct(v):
    """Refinitiv suele dar rendimientos en % (4.25) → decimal."""
    v = num(v)
    return v/100 if pd.notna(v) and abs(v) > 1 else v

def get_history(ric, start, end=None):
    if API is None: return pd.Series(dtype=float)
    end = end or HOY.isoformat()
    for intento in (["TRDPRC_1"], ["TR.PriceClose"], None):
        try:
            if API == "rd":
                df = rd.get_history(universe=ric, fields=intento, interval="daily", start=start, end=end) if intento \
                     else rd.get_history(universe=ric, interval="daily", start=start, end=end)
            else:
                df = ek.get_timeseries(ric, fields="CLOSE", start_date=start, end_date=end)
            df = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
            if not df.empty:
                s = df.iloc[:, 0].dropna(); s.index = pd.to_datetime(s.index); return s
        except Exception as e:
            print(f"  · historia {ric} {intento}: {str(e)[:80]}")
    return pd.Series(dtype=float)

def registra(clave, valor, fuente):
    FUENTES[clave] = fuente; return valor

## 3 · Motor de valuación (Black-Scholes-Merton, bonos, cupón a la par)

In [ ]:
def bsm(S, K, r, sig, T, q=0.0):
    S = np.asarray(S, float)
    sq = sig*np.sqrt(T)
    d1 = (np.log(S/K) + (r - q + 0.5*sig**2)*T)/sq; d2 = d1 - sq
    dq, dr = np.exp(-q*T), np.exp(-r*T)
    c = S*dq*norm.cdf(d1) - K*dr*norm.cdf(d2)
    p = c - S*dq + K*dr                                   # paridad put-call
    return dict(call=c, put=p, d1=d1, d2=d2,
                delta_c=dq*norm.cdf(d1), delta_p=dq*(norm.cdf(d1)-1),
                gamma=dq*norm.pdf(d1)/(S*sq),
                vega=S*dq*norm.pdf(d1)*np.sqrt(T)/100,          # por 1 pt de vol
                theta_c=(-S*dq*norm.pdf(d1)*sig/(2*np.sqrt(T)) - r*K*dr*norm.cdf(d2) + q*S*dq*norm.cdf(d1))/365,
                rho_c=K*T*dr*norm.cdf(d2)/100)                 # por 1 pt de tasa

def flujos(cupon, T, face=1000, f=2):
    n = int(np.ceil(T*f - 1e-9))
    t = np.sort(T - np.arange(n)/f); t = t[t > 1e-9]
    cf = np.full(len(t), face*cupon/f); cf[-1] += face
    return t, cf

def valor_bono(cupon, y, T, face=1000, f=2, limpio=True):
    t, cf = flujos(cupon, T, face, f)
    sucio = np.sum(cf/(1+y/f)**(f*t))
    if not limpio: return sucio
    acum = face*cupon/f*(1 - t[0]*f) if t[0] < 1/f else 0.0
    return sucio - acum

def ytm_de_precio(precio_100, cupon, T, f=2):
    return brentq(lambda y: valor_bono(cupon, y, T, 100, f) - precio_100, -0.05, 0.5)

def dur_conv(cupon, y, T, f=2):
    t, cf = flujos(cupon, T, 100, f); pv = cf/(1+y/f)**(f*t); P = pv.sum()
    mac = np.sum(t*pv)/P; mod = mac/(1+y/f)
    cvx = np.sum(t*(t+1/f)*pv)/(P*(1+y/f)**2)
    return mac, mod, cvx

def cupon_para_valor(objetivo, y, T, face=1000, f=2):
    v0 = valor_bono(0.0, y, T, face, f, False); v1 = valor_bono(0.01, y, T, face, f, False)
    return (objetivo - v0)/(v1 - v0)*0.01

def valuar_convertible(S, prima, r, sig, T, y, q=0.0, face=1000):
    K = S*(1+prima); n = face/K
    o = bsm(S, K, r, sig, T, q)
    opc = o["call"]*n; piso = face - opc
    return dict(K=K, n=n, call_acc=float(o["call"]), put_acc=float(o["put"]), delta=float(o["delta_c"]),
                opcion_bono=float(opc), piso=float(piso), cupon=cupon_para_valor(piso, y, T, face), greeks=o)

## 4 · Validación: replicar el caso Molson Coors (Exhibits 2, 3 y 4)

In [ ]:
m = valuar_convertible(87.61, 0.25, 0.0446, 0.23, 6, 0.0575, q=0.0)
w = bsm(87.61, 87.61*1.60, 0.0446, 0.21, 6)
VALID = pd.DataFrame([
    ["Precio de conversión",       109.51, m["K"]],
    ["Acciones por bono",          9.13,   m["n"]],
    ["Call por acción (25%)",      21.0,   m["call_acc"]],
    ["Delta call",                 0.64,   m["delta"]],
    ["Put por paridad",            17.2,   m["put_acc"]],
    ["Opción de conversión/bono",  191.5,  m["opcion_bono"]],
    ["Valor bono puro",            808.5,  m["piso"]],
    ["Cupón a la par (%)",         1.93,   m["cupon"]*100],
    ["Warrant 60% por acción",     11.4,   float(w["call"])],
    ["Costo note hedge (USD M)",   110.0,  m["call_acc"]*5.25],
    ["Ingreso warrants (USD M)",   59.0,   float(w["call"])*5.25],
], columns=["Concepto", "Caso", "Modelo"])
VALID["Diferencia"] = VALID["Modelo"] - VALID["Caso"]
VALID

## 5 · Universo de bonos de Apple en Refinitiv y selección

In [ ]:
def buscar_bonos_aapl():
    if API != "rd": return pd.DataFrame()
    sel = "RIC,ISIN,CUSIP,IssuerLegalName,IssuerCommonName,CouponRate,MaturityDate,IssueDate,FaceIssuedTotal,DebtTypeDescription,RCSCurrencyLeaf"
    for flt in ["IsActive eq true and RCSCurrencyLeaf eq 'US Dollar'", "IsActive eq true", None]:
        try:
            kw = dict(view=rd.discovery.Views.GOV_CORP_INSTRUMENTS, query="Apple Inc", select=sel, top=300)
            if flt: kw["filter"] = flt
            df = rd.discovery.search(**kw)
            if df is not None and not df.empty:
                txt = df.astype(str).agg(" ".join, axis=1)
                df = df[txt.str.contains("Apple Inc", case=False)]
                if "RCSCurrencyLeaf" in df: df = df[df["RCSCurrencyLeaf"].astype(str).str.contains("US Dollar")]
                if not df.empty: return df
        except Exception as e:
            print("  · discovery.search:", str(e)[:100])
    return pd.DataFrame()

raw = buscar_bonos_aapl()
if not raw.empty:
    U = pd.DataFrame({
        "nombre": "AAPL " + raw.get("CouponRate", pd.Series(index=raw.index)).astype(str) + "% " + raw.get("MaturityDate", "").astype(str).str[:10],
        "ric": raw.get("RIC"), "isin": raw.get("ISIN"), "cusip": raw.get("CUSIP"),
        "cupon": pd.to_numeric(raw.get("CouponRate"), errors="coerce")/100,
        "vencimiento": pd.to_datetime(raw.get("MaturityDate"), errors="coerce").dt.date,
        "monto": pd.to_numeric(raw.get("FaceIssuedTotal"), errors="coerce"),
    })
    FUENTE_UNIV = "Refinitiv discovery.search (GOV_CORP_INSTRUMENTS)"
else:
    U = pd.DataFrame(UNIVERSO_RESPALDO); U["vencimiento"] = pd.to_datetime(U["vencimiento"]).dt.date
    U["ric"] = U["cusip"] + "="; U["monto"] = np.nan
    FUENTE_UNIV = "⚠ respaldo (term sheets SEC)"

U["ric"] = U["ric"].fillna(U["cusip"].astype(str) + "=")
U = U.dropna(subset=["cupon", "vencimiento"])
U = U[pd.to_datetime(U["vencimiento"]) > pd.Timestamp(HOY)].drop_duplicates("isin").sort_values("vencimiento").head(40)
U["anios"] = (pd.to_datetime(U["vencimiento"]) - pd.Timestamp(HOY)).dt.days/365.25

# Precio y rendimiento de cada bono
precios = get_data(U["ric"].tolist(), ["MID_PRICE"]) if API else pd.DataFrame()
if not precios.empty and precios.shape[1] >= 2:
    U["precio"] = pd.to_numeric(precios.iloc[:, -1].values, errors="coerce")
else:
    U["precio"] = np.nan
U["ytm"] = [ytm_de_precio(p, c, T) if pd.notna(p) and p > 0 else np.nan for p, c, T in zip(U["precio"], U["cupon"], U["anios"])]

# Puntaje de idoneidad para replicar el caso: plazo ≈ 6 años, precio ≈ par, bullet senior
U["dist_plazo"] = (U["anios"] - PLAZO_CONV).abs()
U["dist_par"]   = (U["precio"].fillna(100) - 100).abs()
U["puntaje"]    = (100 - 12*U["dist_plazo"] - 2*U["dist_par"]).round(1)
print("Fuente universo:", FUENTE_UNIV)
U.sort_values("puntaje", ascending=False)[["nombre","cusip","isin","cupon","vencimiento","anios","precio","ytm","puntaje"]]

## 6 · Datos de mercado del bono elegido, Treasuries y acción

In [ ]:
T_BONO = (pd.Timestamp(BONO["vencimiento"]) - pd.Timestamp(HOY)).days/365.25
ids_bono = [BONO["cusip"] + "=", BONO["isin"]]

precio_bono = fuente_p = None
for i in ids_bono:
    precio_bono, fuente_p = first_value(i, ["MID_PRICE", "CF_CLOSE", "CF_LAST", "TR.FiClosePrice", "TR.MidPrice"])
    if precio_bono is not None: break
if precio_bono is None:
    precio_bono = registra("precio_bono", RESPALDO["precio_bono"], "⚠ respaldo")
else:
    precio_bono = registra("precio_bono", num(precio_bono), f"Refinitiv {fuente_p}")

YTM = registra("ytm", ytm_de_precio(precio_bono, BONO["cupon"], T_BONO), f"Calculado de precio ({FUENTES['precio_bono']})")
y_ref, f_y = (None, None)
for i in ids_bono:
    y_ref, f_y = first_value(i, ["MID_YLD_1", "RT_YIELD_1", "SEC_YLD_1", "TR.MidYield"])
    if y_ref is not None: break
YTM_REF = pct(y_ref) if y_ref is not None else np.nan
MAC, MOD, CVX = dur_conv(BONO["cupon"], YTM, T_BONO)

# Curva Treasury
UST = {}
for plazo in [2, 3, 5, 7, 10, 30]:
    v, fld = first_value(f"US{plazo}YT=RR", ["MID_YLD_1", "B_YLD_1", "RT_YIELD_1", "TR.MidYield"])
    UST[plazo] = pct(v) if v is not None else RESPALDO["ust"][plazo]
    FUENTES[f"ust{plazo}"] = f"Refinitiv {fld}" if v is not None else "⚠ respaldo"
ust_x, ust_y = zip(*sorted(UST.items()))
rf = lambda T: float(np.interp(T, ust_x, ust_y))
RF = registra("rf", rf(PLAZO_CONV), f"UST interpolado a {PLAZO_CONV:.0f}a ({FUENTES['ust5']})")
SPREAD = YTM - rf(T_BONO)

# Acción
S, fs = first_value(TICKER_RIC, ["CF_LAST", "TRDPRC_1", "TR.PriceClose"])
S = registra("S", num(S), f"Refinitiv {fs}") if S is not None else registra("S", RESPALDO["S"], "⚠ respaldo")

HIST = get_history(TICKER_RIC, (HOY - dt.timedelta(days=5*365+10)).isoformat())
if not HIST.empty:
    lr = np.log(HIST).diff().dropna()
    VOLS = {k: lr[lr.index >= lr.index[-1] - pd.Timedelta(days=365*int(k[0]))].std()*np.sqrt(252) for k in ["1Y", "3Y", "5Y"]}
    VOL_ROLL = lr.rolling(63).std().dropna()*np.sqrt(252)
else:
    VOLS = {"1Y": RESPALDO["vol"], "3Y": RESPALDO["vol"], "5Y": RESPALDO["vol"]}; VOL_ROLL = pd.Series(dtype=float)
if VOL_MANUAL:
    SIGMA = registra("vol", VOL_MANUAL, "Manual (implícita)")
else:
    SIGMA = registra("vol", float(VOLS[VENTANA_VOL]), f"Histórica {VENTANA_VOL} Refinitiv" if not HIST.empty else "⚠ respaldo")

dy, fdy = first_value(TICKER_RIC, ["TR.DividendYield", "YIELD"])
Q = registra("q", pct(dy), f"Refinitiv {fdy}") if dy is not None else registra("q", RESPALDO["div_yield"], "⚠ respaldo")
Q_USO = Q if USAR_DIVIDENDO else 0.0
MONTO = MONTO_EMISION or BONO["monto_usd"]

pd.Series({"Precio limpio bono": precio_bono, "YTM calculado": YTM, "YTM Refinitiv": YTM_REF,
           "Años al vencimiento": T_BONO, "Duración Macaulay": MAC, "Duración modificada": MOD, "Convexidad": CVX,
           "rf (UST interp. 6a)": RF, "Spread vs UST": SPREAD, "AAPL spot": S, "Vol usada": SIGMA,
           "Vol 1Y/3Y/5Y": " / ".join(f"{v:.1%}" for v in VOLS.values()), "Dividend yield": Q})

## 7 · Empresa (KPIs estilo dashboard FINA 405) y contexto China

In [ ]:
CAMPOS_EMP = {
    "Market Cap":        ["TR.CompanyMarketCap"],
    "Revenue (LTM)":     ["TR.Revenue"],
    "EBITDA":            ["TR.EBITDA"],
    "Net Income":        ["TR.NetIncome", "TR.NetIncomeBeforeExtraItems"],
    "Total Debt":        ["TR.TotalDebtOutstanding", "TR.TotalDebt"],
    "Total Assets":      ["TR.TotalAssetsReported", "TR.TotalAssets"],
    "Total Equity":      ["TR.TotalEquity", "TR.ShareholdersEquity"],
    "Cash & ST Inv.":    ["TR.CashAndSTInvestments", "TR.CashandEquivalents"],
    "Interest Expense":  ["TR.InterestExpense", "TR.IntExpense"],
    "EBIT":              ["TR.EBIT"],
    "Shares Out.":       ["TR.SharesOutstanding"],
}
EMP = {}
for k, flds in CAMPOS_EMP.items():
    v, f = first_value(TICKER_RIC, flds)
    EMP[k] = num(v) if v is not None else np.nan
    FUENTES[f"emp_{k}"] = f"Refinitiv {f}" if f else "n/d"

def ratio(a, b): return EMP.get(a, np.nan)/EMP.get(b, np.nan) if pd.notna(EMP.get(b)) and EMP.get(b) else np.nan
EMP["D/E"]               = ratio("Total Debt", "Total Equity")
EMP["Debt/Assets"]       = ratio("Total Debt", "Total Assets")
EMP["Interest Coverage"] = abs(ratio("EBIT", "Interest Expense"))
EMP["Net Debt/EBITDA"]   = (EMP["Total Debt"] - EMP["Cash & ST Inv."])/EMP["EBITDA"] if pd.notna(EMP["EBITDA"]) else np.nan
EMP["EBIT Margin"]       = ratio("EBIT", "Revenue (LTM)")
EMP["Net Margin"]        = ratio("Net Income", "Revenue (LTM)")

# Dilución potencial del convertible sintético
EMP["Dilución conv. 25%"] = (MONTO/(S*(1+PRIMA_CONV)))/EMP["Shares Out."] if pd.notna(EMP["Shares Out."]) else np.nan

PAIS = {}
for etiqueta, ric, flds in [("USD/CNY", "CNY=", ["CF_LAST", "MID_PRICE"]), ("China 10Y", "CN10YT=RR", ["MID_YLD_1", "B_YLD_1"]),
                            ("US 10Y", "US10YT=RR", ["MID_YLD_1", "B_YLD_1"])]:
    v, f = first_value(ric, flds)
    PAIS[etiqueta] = (pct(v) if "Y" in etiqueta and v is not None else num(v)) if v is not None else np.nan
pd.Series({**EMP, **PAIS})

## 8 · Valuación del convertible sintético AAPL (Exhibits 2 y 3)

In [ ]:
C = valuar_convertible(S, PRIMA_CONV, RF, SIGMA, PLAZO_CONV, YTM, q=Q_USO)
C0 = valuar_convertible(S, PRIMA_CONV, RF, SIGMA, PLAZO_CONV, YTM, q=0.0)     # versión caso (sin dividendo)
g = C["greeks"]

EXH2 = pd.DataFrame({
    "Por acción": [S, C["K"], RF, SIGMA, PLAZO_CONV, Q_USO, C["call_acc"], C["delta"], C["put_acc"], float(g["gamma"]), float(g["vega"]), float(g["theta_c"]), float(g["rho_c"])],
    "Por bono":   [C["n"]*S, 1000, RF, SIGMA, PLAZO_CONV, Q_USO, C["opcion_bono"], C["delta"], C["put_acc"]*C["n"], float(g["gamma"])/C["n"], float(g["vega"])*C["n"], float(g["theta_c"])*C["n"], float(g["rho_c"])*C["n"]],
}, index=["S", "X", "rf", "σ", "T", "q", "Call", "Delta", "Put (paridad)", "Gamma", "Vega (1pt)", "Theta (día)", "Rho (1pt)"])

t_cf, cf = flujos(C["cupon"], PLAZO_CONV)
pv = cf/(1+YTM/2)**(2*t_cf)
EXH3 = pd.DataFrame({"Año": t_cf, "Cupón": 1000*C["cupon"]/2, "Principal": np.where(t_cf == t_cf[-1], 1000, 0), "Flujo": cf, "VP": pv})

print(f"Precio de conversión ${C['K']:.2f} | acciones/bono {C['n']:.4f}")
print(f"Opción de conversión por bono ${C['opcion_bono']:.2f} (sin dividendo: ${C0['opcion_bono']:.2f})")
print(f"Bono puro ${C['piso']:.2f} → cupón a la par {C['cupon']:.3%} (sin dividendo: {C0['cupon']:.3%})")
print(f"Cupón straight (YTM) {YTM:.3%} → ahorro {(YTM-C['cupon'])*1e4:.0f} pb = USD {(YTM-C['cupon'])*MONTO/1e6:,.1f} M anuales")
EXH2

## 9 · Sensibilidades: prima de conversión y volatilidad

In [ ]:
SENS = pd.DataFrame([{**{"Prima": p}, **{k: v for k, v in valuar_convertible(S, p, RF, SIGMA, PLAZO_CONV, YTM, Q_USO).items() if k in ("K", "opcion_bono", "piso", "cupon", "delta")}} for p in PRIMAS_SENS])
SENS["Ahorro (pb)"] = (YTM - SENS["cupon"])*1e4
SENS["Ahorro anual USD M"] = (YTM - SENS["cupon"])*MONTO/1e6
SENS["Cupón negativo?"] = SENS["cupon"] < 0

VOL_GRID = np.round(np.arange(max(0.10, SIGMA-0.10), SIGMA+0.105, 0.025), 3)
HEAT = pd.DataFrame([[valuar_convertible(S, p, RF, v, PLAZO_CONV, YTM, Q_USO)["cupon"]*100 for v in VOL_GRID] for p in PRIMAS_SENS],
                    index=[f"{p:.0%}" for p in PRIMAS_SENS], columns=[f"{v:.1%}" for v in VOL_GRID])
SENS

## 10 · Call spread overlay: note hedge 25% + warrant 60% (Exhibits 1 y 4)

In [ ]:
K1, K2 = S*(1+PRIMA_CONV), S*(1+PRIMA_WARRANT)
N_ACC = MONTO/K1
VOL_W = SIGMA + AJUSTE_VOL_WARR
hedge = bsm(S, K1, RF, SIGMA, PLAZO_CONV, Q_USO)
warr  = bsm(S, K2, RF, VOL_W, PLAZO_CONV, Q_USO)
COSTO_HEDGE, INGRESO_WARR = float(hedge["call"])*N_ACC, float(warr["call"])*N_ACC
NETO = COSTO_HEDGE - INGRESO_WARR

# "No free lunch": emitir directo al 60% exige un cupón aún menor; compara VP del cupón extra vs costo del spread
C60 = valuar_convertible(S, PRIMA_WARRANT, RF, SIGMA, PLAZO_CONV, YTM, Q_USO)
VP_EXTRA = (valor_bono(C60["cupon"], YTM, PLAZO_CONV, 1000, 2, False) - valor_bono(C["cupon"], YTM, PLAZO_CONV, 1000, 2, False))*MONTO/1000

EXH4 = pd.DataFrame({
    "% del principal": [COSTO_HEDGE/MONTO, INGRESO_WARR/MONTO, NETO/MONTO],
    "USD M":           [COSTO_HEDGE/1e6, INGRESO_WARR/1e6, NETO/1e6],
    "Caso Molson %":   [0.186, 0.099, 0.086],
}, index=["Costo note hedge (compra calls K=25%)", "Ingreso warrants (venta calls K=60%)", "Costo neto call spread"])
print(f"Acciones subyacentes {N_ACC/1e6:.2f} M | precio conversión efectivo ${K2:.2f} (prima efectiva {PRIMA_WARRANT:.0%})")
print(f"Cupón directo al 60%: {C60['cupon']:.3%} | VP de pagar cupón 25% vs 60%: USD {VP_EXTRA/1e6:,.1f} M vs costo neto spread USD {NETO/1e6:,.1f} M")
EXH4

## 11 · Gráficas

In [ ]:
PAL = dict(azul="#1f4e79", azul2="#2e75b6", verde="#2e7d32", rojo="#c62828", gris="#6b7280", oro="#b7791f")
LAY = dict(template="plotly_white", font=dict(family="Inter, Segoe UI, Arial", size=12), margin=dict(l=50, r=20, t=50, b=40), height=380,
           legend=dict(orientation="h", y=-0.2))
FIGS = {}

# 1. Precio AAPL con precios de conversión
f = go.Figure()
if not HIST.empty: f.add_trace(go.Scatter(x=HIST.index, y=HIST.values, name="AAPL", line=dict(color=PAL["azul"])))
for val, nom, col in [(S, "Spot", PAL["gris"]), (K1, f"Conversión {PRIMA_CONV:.0%}", PAL["verde"]), (K2, f"Warrant {PRIMA_WARRANT:.0%}", PAL["rojo"])]:
    f.add_hline(y=val, line_dash="dash", line_color=col, annotation_text=f"{nom} ${val:,.0f}")
f.update_layout(title="AAPL — precio histórico vs precios de conversión", **LAY); FIGS["precio"] = f

# 2. Curva de bonos Apple vs UST
f = go.Figure()
f.add_trace(go.Scatter(x=list(ust_x), y=np.array(ust_y)*100, name="UST", mode="lines+markers", line=dict(color=PAL["gris"])))
UU = U.dropna(subset=["ytm"])
f.add_trace(go.Scatter(x=UU["anios"], y=UU["ytm"]*100, mode="markers", name="Bonos AAPL", text=UU["nombre"], marker=dict(color=PAL["azul2"], size=9)))
f.add_trace(go.Scatter(x=[T_BONO], y=[YTM*100], mode="markers", name="Bono elegido", marker=dict(color=PAL["rojo"], size=15, symbol="star")))
f.update_layout(title="Curva USD de Apple vs Treasuries", xaxis_title="Años al vencimiento", yaxis_title="Rendimiento (%)", **LAY); FIGS["curva"] = f

# 3. Flujos del bono puro (Exhibit 3)
f = go.Figure()
f.add_trace(go.Bar(x=EXH3["Año"], y=EXH3["Flujo"], name="Flujo", marker_color=PAL["azul2"]))
f.add_trace(go.Bar(x=EXH3["Año"], y=EXH3["VP"], name="Valor presente", marker_color=PAL["oro"]))
f.update_layout(title=f"Bono puro: cupón {C['cupon']:.2%} descontado al {YTM:.2%} → ${C['piso']:.1f}", barmode="group", yaxis_type="log", xaxis_title="Año", **LAY); FIGS["flujos"] = f

# 4. Valor del convertible vs precio de la acción
Sx = np.linspace(0.3*S, 2.5*S, 200)
conv_hoy = C["piso"] + C["n"]*bsm(Sx, C["K"], RF, SIGMA, PLAZO_CONV, Q_USO)["call"]
f = go.Figure()
f.add_trace(go.Scatter(x=Sx, y=np.full_like(Sx, C["piso"]), name="Piso (bono puro)", line=dict(color=PAL["gris"], dash="dash")))
f.add_trace(go.Scatter(x=Sx, y=C["n"]*Sx, name="Valor de conversión (paridad)", line=dict(color=PAL["verde"], dash="dot")))
f.add_trace(go.Scatter(x=Sx, y=conv_hoy, name="Convertible (bono + call)", line=dict(color=PAL["azul"], width=3)))
f.add_vline(x=S, line_color=PAL["rojo"], annotation_text="Spot")
f.update_layout(title="Convertible = bono puro + bundle de calls", xaxis_title="Precio AAPL", yaxis_title="USD por bono de $1,000", **LAY); FIGS["convertible"] = f

# 5. Cupón vs prima
f = make_subplots(specs=[[{"secondary_y": True}]])
f.add_trace(go.Bar(x=[f"{p:.0%}" for p in SENS["Prima"]], y=SENS["opcion_bono"], name="Opción por bono (USD)", marker_color=PAL["azul2"]), secondary_y=False)
f.add_trace(go.Scatter(x=[f"{p:.0%}" for p in SENS["Prima"]], y=SENS["cupon"]*100, name="Cupón a la par (%)", line=dict(color=PAL["rojo"], width=3)), secondary_y=True)
f.add_hline(y=YTM*100, line_dash="dash", line_color=PAL["gris"], secondary_y=True, annotation_text="YTM straight")
f.update_layout(title="Menor prima → opción más valiosa → cupón más bajo", **LAY); FIGS["prima"] = f

# 6. Heatmap
f = go.Figure(go.Heatmap(z=HEAT.values, x=HEAT.columns, y=HEAT.index, colorscale="RdYlGn_r", text=np.round(HEAT.values, 2), texttemplate="%{text}%", colorbar=dict(title="Cupón %")))
f.update_layout(title="Cupón a la par: prima de conversión × volatilidad", xaxis_title="Volatilidad", yaxis_title="Prima", **{**LAY, "height": 420}); FIGS["heat"] = f

# 7. Call spread payoff (por acción subyacente, al vencimiento)
ST = np.linspace(0.5*S, 2.5*S, 200)
pay = np.maximum(ST-K1, 0) - np.maximum(ST-K2, 0)
f = go.Figure()
f.add_trace(go.Scatter(x=ST, y=pay, name="Payoff call spread (Apple)", line=dict(color=PAL["verde"], width=3)))
f.add_trace(go.Scatter(x=ST, y=pay - NETO/N_ACC*np.exp(RF*PLAZO_CONV), name="Neto de costo (valor futuro)", line=dict(color=PAL["azul"], dash="dash")))
f.add_trace(go.Scatter(x=ST, y=np.maximum(ST-K1, 0), name="Dilución sin overlay", line=dict(color=PAL["rojo"], dash="dot")))
f.update_layout(title="Overlay: compra call 25% + venta warrant 60%", xaxis_title="AAPL al vencimiento", yaxis_title="USD por acción", **LAY); FIGS["spread"] = f

# 8. Griegas por bono vs S
gx = bsm(Sx, C["K"], RF, SIGMA, PLAZO_CONV, Q_USO)
f = make_subplots(specs=[[{"secondary_y": True}]])
f.add_trace(go.Scatter(x=Sx, y=gx["delta_c"], name="Delta", line=dict(color=PAL["azul"])), secondary_y=False)
f.add_trace(go.Scatter(x=Sx, y=gx["gamma"]*C["n"]*S, name="Gamma × n × S", line=dict(color=PAL["oro"])), secondary_y=True)
f.add_trace(go.Scatter(x=Sx, y=gx["vega"]*C["n"], name="Vega por bono", line=dict(color=PAL["verde"], dash="dot")), secondary_y=True)
f.update_layout(title="Griegas de la opción de conversión", xaxis_title="Precio AAPL", **LAY); FIGS["griegas"] = f

# 9. Volatilidad móvil
f = go.Figure()
if not VOL_ROLL.empty: f.add_trace(go.Scatter(x=VOL_ROLL.index, y=VOL_ROLL*100, name="Vol 3M móvil", line=dict(color=PAL["azul2"])))
f.add_hline(y=SIGMA*100, line_dash="dash", line_color=PAL["rojo"], annotation_text=f"σ usada {SIGMA:.1%}")
f.update_layout(title="Volatilidad realizada de AAPL", yaxis_title="%", **LAY); FIGS["vol"] = f
FIGS["convertible"].show()

## 12 · Dashboard HTML (mix: formato FINA 405 + caso Molson)

In [ ]:
def fm(v, tipo="num", d=2):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "n/d"
    if tipo == "pct": return f"{v*100:,.{d}f}%"
    if tipo == "usd": return f"${v:,.{d}f}"
    if tipo == "bn":  return f"${v/1e9:,.1f} B"
    if tipo == "x":   return f"{v:,.{d}f}x"
    return f"{v:,.{d}f}"

def card(titulo, valor, sub="", alerta=False):
    return f'<div class="card{" warn" if alerta else ""}"><div class="lab">{titulo}</div><div class="val">{valor}</div><div class="sub">{sub}</div></div>'

def fuente(k): return FUENTES.get(k, "")
def esresp(k): return "respaldo" in FUENTES.get(k, "")
def tabla(df, fmt=None):
    return df.to_html(classes="tbl", float_format=lambda x: f"{x:,.4f}", border=0, na_rep="n/d") if fmt is None else df.style.format(fmt).hide(axis="index").to_html()
def fig(k): return FIGS[k].to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False})

ahorro_pb = (YTM - C["cupon"])*1e4
concl = f"""
<p><b>Bono base.</b> {BONO['nombre']} ({BONO['isin']}): precio {precio_bono:.3f}, YTM {YTM:.2%}, spread {SPREAD*1e4:.0f} pb sobre UST,
duración modificada {MOD:.2f}. Es el análogo directo del bono a 6 años de Molson (tasa de descuento 5.75% en el caso).</p>
<p><b>Convertible sintético a {PLAZO_CONV:.0f} años, prima {PRIMA_CONV:.0%}.</b> La opción de conversión vale ${C['opcion_bono']:.1f} por bono
({C['n']:.3f} acciones a ${C['K']:.2f}); el bono puro debe valer ${C['piso']:.1f}, lo que exige un cupón de <b>{C['cupon']:.2%}</b> frente a {YTM:.2%}
de deuda straight: ahorro de {ahorro_pb:.0f} pb, ≈ USD {(YTM-C['cupon'])*MONTO/1e6:,.1f} M anuales sobre USD {MONTO/1e9:.1f} B.</p>
<p><b>Visión del CEO.</b> Con σ = {SIGMA:.1%} (vs 23% de Molson) y dividendo de {Q:.2%}, el valor de la opción es {'mayor' if SIGMA > 0.23 else 'menor'} que en el caso,
por lo que el cupón resultante {'puede volverse negativo' if C['cupon'] < 0 else 'es muy bajo'}. Una prima de 25% implica dilución de
{fm(EMP.get('Dilución conv. 25%'), 'pct')} de las acciones y una conversión probable si AAPL mantiene su tendencia; por eso, igual que Molson, conviene elevar la prima efectiva.</p>
<p><b>Call spread.</b> Comprar calls al {PRIMA_CONV:.0%} cuesta {COSTO_HEDGE/MONTO:.1%} del principal y vender warrants al {PRIMA_WARRANT:.0%} recupera {INGRESO_WARR/MONTO:.1%}:
costo neto {NETO/MONTO:.1%} (caso: 8.6%). Emitir directo al 60% requeriría cupón {C60['cupon']:.2%}; el VP del cupón extra (USD {VP_EXTRA/1e6:,.1f} M) frente al costo neto
del spread (USD {NETO/1e6:,.1f} M) confirma el punto del caso: <i>no hay free lunch</i>, la prima más alta siempre se paga.</p>
<p><b>Nota.</b> El bono real tiene make-whole call (Molson era non-call life); a ese spread de Treasury la opción de redención es casi nula y se ignora,
igual que en el caso. Apple no tiene convertibles: el ejercicio es una emisión hipotética.</p>
"""

kpi_resumen = "".join([
    card("Bono base", "4.500% 2032", BONO["isin"]),
    card("Precio limpio", fm(precio_bono, d=3), fuente("precio_bono"), esresp("precio_bono")),
    card("YTM (tasa descuento)", fm(YTM, "pct"), "Equivalente al 5.75% del caso"),
    card("Spread vs UST", f"{SPREAD*1e4:,.0f} pb", f"UST {T_BONO:.1f}a {rf(T_BONO):.2%}"),
    card("Duración modificada", fm(MOD), f"Convexidad {CVX:.1f}"),
    card("AAPL spot", fm(S, "usd"), fuente("S"), esresp("S")),
    card("Volatilidad σ", fm(SIGMA, "pct", 1), fuente("vol"), esresp("vol")),
    card("rf 6 años", fm(RF, "pct"), fuente("rf"), esresp("ust5")),
    card("Opción conversión/bono", fm(C["opcion_bono"], "usd", 1), f"{C['n']:.3f} acc. × ${C['call_acc']:.2f}"),
    card("Bono puro", fm(C["piso"], "usd", 1), "Piso del convertible"),
    card("Cupón a la par", fm(C["cupon"], "pct"), f"Ahorro {ahorro_pb:,.0f} pb vs straight"),
    card("Call spread neto", fm(NETO/MONTO, "pct", 1), f"USD {NETO/1e6:,.1f} M · caso 8.6%"),
])
kpi_emp = "".join([card(k, fm(EMP[k], t), "Refinitiv" if "Refinitiv" in FUENTES.get(f"emp_{k}", "x Refinitiv") else "n/d") for k, t in [
    ("Market Cap", "bn"), ("Revenue (LTM)", "bn"), ("EBITDA", "bn"), ("Net Income", "bn"), ("Total Debt", "bn"), ("Cash & ST Inv.", "bn"),
    ("D/E", "x"), ("Debt/Assets", "pct"), ("Interest Coverage", "x"), ("Net Debt/EBITDA", "x"), ("EBIT Margin", "pct"), ("Net Margin", "pct")]])
kpi_pais = "".join([card("USD/CNY", fm(PAIS["USD/CNY"], d=4), "Spot"), card("China 10Y", fm(PAIS["China 10Y"], "pct"), "Gov. yield"),
                    card("US 10Y", fm(PAIS["US 10Y"], "pct"), "Treasury"), card("Diferencial CN−US", fm(PAIS["China 10Y"]-PAIS["US 10Y"], "pct"), "Puntos de tasa")])

tabs = [
 ("Resumen", f'<div class="grid">{kpi_resumen}</div><div class="two">{fig("convertible")}{fig("precio")}</div>'),
 ("Bonos AAPL", f'<p class="nota">Fuente: {FUENTE_UNIV}. Puntaje = cercanía a 6 años y a precio par (criterios del caso).</p>{fig("curva")}' +
     tabla(U.sort_values("puntaje", ascending=False)[["nombre","cusip","isin","cupon","vencimiento","anios","precio","ytm","puntaje"]].reset_index(drop=True))),
 ("Empresa & País", f'<h3>Apple Inc. — perfil financiero</h3><div class="grid">{kpi_emp}</div><h3>Contexto China (país asignado)</h3><div class="grid">{kpi_pais}</div>'),
 ("Opción de conversión", f'<h3>Exhibit 2 — Black-Scholes{"-Merton (q=" + format(Q_USO, ".2%") + ")" if Q_USO else ""}</h3>{tabla(EXH2)}<div class="two">{fig("griegas")}{fig("vol")}</div>'),
 ("Bono puro & cupón", f'<h3>Exhibit 3 — Bono puro y valor del convertible</h3>{fig("flujos")}{tabla(EXH3)}'
     f'<p class="nota">Convertible = ${C["piso"]:.2f} + ${C["opcion_bono"]:.2f} = ${C["piso"]+C["opcion_bono"]:.2f}</p>'),
 ("Sensibilidades", f'<div class="two">{fig("prima")}{fig("heat")}</div>{tabla(SENS)}'),
 ("Call spread", f'<h3>Exhibits 1 y 4 — Note hedge + warrants</h3>{tabla(EXH4)}{fig("spread")}'
     f'<p class="nota">Cupón directo al {PRIMA_WARRANT:.0%}: {C60["cupon"]:.3%} · VP del cupón extra USD {VP_EXTRA/1e6:,.1f} M vs costo neto USD {NETO/1e6:,.1f} M · σ warrant {VOL_W:.1%}</p>'),
 ("Validación Molson", f'<p class="nota">El modelo reproduce los exhibits del caso con sus mismos supuestos.</p>{tabla(VALID.set_index("Concepto"))}'),
 ("Conclusiones", concl),
]
nav  = "".join(f'<button class="tab{" on" if i==0 else ""}" onclick="ver({i})">{t}</button>' for i, (t, _) in enumerate(tabs))
body = "".join(f'<section id="s{i}" style="display:{"block" if i==0 else "none"}">{c}</section>' for i, (_, c) in enumerate(tabs))
fuentes_txt = " · ".join(f"{k}: {v}" for k, v in FUENTES.items() if not k.startswith("emp_"))

HTML = f"""<!doctype html><html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>AAPL Convertible Sintético</title><script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
:root{{--bg:#f5f7fa;--card:#fff;--ink:#1a202c;--mut:#6b7280;--acc:#1f4e79;--warn:#b45309}}
body{{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,Segoe UI,Arial,sans-serif}}
header{{background:linear-gradient(135deg,#0f2742,#1f4e79);color:#fff;padding:22px 28px}}
header h1{{margin:0;font-size:22px}} header p{{margin:4px 0 0;opacity:.85;font-size:13px}}
nav{{display:flex;flex-wrap:wrap;gap:6px;padding:12px 28px;background:#fff;border-bottom:1px solid #e5e7eb;position:sticky;top:0;z-index:5}}
.tab{{border:1px solid #d1d5db;background:#fff;border-radius:18px;padding:6px 14px;cursor:pointer;font-size:13px}}
.tab.on{{background:var(--acc);color:#fff;border-color:var(--acc)}}
main{{padding:20px 28px;max-width:1400px;margin:auto}}
.grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(200px,1fr));gap:12px;margin-bottom:18px}}
.card{{background:var(--card);border-radius:10px;padding:14px 16px;box-shadow:0 1px 3px rgba(0,0,0,.08);border-left:4px solid var(--acc)}}
.card.warn{{border-left-color:var(--warn)}} .lab{{font-size:12px;color:var(--mut);text-transform:uppercase;letter-spacing:.04em}}
.val{{font-size:22px;font-weight:700;margin:4px 0}} .sub{{font-size:11px;color:var(--mut)}}
.two{{display:grid;grid-template-columns:repeat(auto-fit,minmax(480px,1fr));gap:14px}}
.two>div,section>div.plotly-graph-div{{background:#fff;border-radius:10px;padding:6px}}
table{{border-collapse:collapse;background:#fff;font-size:12.5px;margin:10px 0 18px;width:100%;overflow-x:auto;display:block}}
th,td{{padding:6px 10px;border-bottom:1px solid #eef0f3;text-align:right;white-space:nowrap}} th{{background:#f0f4f8;color:#334155}}
.nota{{font-size:12.5px;color:var(--mut)}} section p{{line-height:1.55;max-width:1050px}}
footer{{font-size:11px;color:var(--mut);padding:16px 28px}}
@media(max-width:600px){{.two{{grid-template-columns:1fr}} main,nav,header{{padding-left:16px;padding-right:16px}}}}
</style></head><body>
<header><h1>Apple Inc. (AAPL) — Bono Convertible Sintético</h1>
<p>{CURSO} · {AUTORES} · Base: caso Molson Coors (NE90-06-107) · Datos: Refinitiv/LSEG · {HOY:%d-%b-%Y}</p></header>
<nav>{nav}</nav><main>{body}</main>
<footer>Fuentes: {fuentes_txt}. Las tarjetas con borde ámbar usan valores de respaldo.</footer>
<script>function ver(i){{document.querySelectorAll("section").forEach((s,j)=>s.style.display=j==i?"block":"none");
document.querySelectorAll(".tab").forEach((b,j)=>b.classList.toggle("on",j==i));window.dispatchEvent(new Event("resize"));}}</script>
</body></html>"""

RUTA = os.path.join(CARPETA, "AAPL_Convertible_Dashboard.html")
with open(RUTA, "w", encoding="utf-8") as fh: fh.write(HTML)
print("Dashboard guardado en:", os.path.abspath(RUTA))
from IPython.display import IFrame, display
display(IFrame(src=RUTA, width="100%", height=900))

## 13 · Exportar tablas a Excel (anexos del reporte)

In [ ]:
try:
    with pd.ExcelWriter(os.path.join(CARPETA, "AAPL_Convertible_Tablas.xlsx")) as xw:
        U.to_excel(xw, sheet_name="Universo_bonos", index=False); EXH2.to_excel(xw, sheet_name="Exhibit2_Opcion")
        EXH3.to_excel(xw, sheet_name="Exhibit3_Bono", index=False); SENS.to_excel(xw, sheet_name="Sensibilidad_prima", index=False)
        HEAT.to_excel(xw, sheet_name="Heatmap_cupon"); EXH4.to_excel(xw, sheet_name="Exhibit4_CallSpread")
        VALID.to_excel(xw, sheet_name="Validacion_Molson", index=False)
        pd.Series(FUENTES).to_frame("Fuente").to_excel(xw, sheet_name="Fuentes")
    print("Excel exportado.")
except Exception as e:
    print("No se exportó Excel (instala openpyxl):", e)